# Quickstart: using the Global Markets OHLCV datasets

Demo notebook for:

| Slice | Dataset | Intervals |
|---|---|---|
| **Daily / weekly** | [benjaminpo/finance-dataset](https://www.kaggle.com/datasets/benjaminpo/finance-dataset) | `1d`, `1wk`, … |
| **Intraday** | [benjaminpo/finance-dataset-intraday](https://www.kaggle.com/datasets/benjaminpo/finance-dataset-intraday) | `1m` … `1h` (dated snapshots) |

It shows how to:
1. Locate both attached dataset roots on Kaggle (or a local `data/` checkout)
# Quickstart: using the Global Markets OHLCV datasets

Demo notebook for:

| Slice | Dataset | Intervals |
|---|---|---|
| **Daily / weekly** | [benjaminpo/finance-dataset](https://www.kaggle.com/datasets/benjaminpo/finance-dataset) | `1d`, `1wk`, … |
| **Intraday** | [benjaminpo/finance-dataset-intraday](https://www.kaggle.com/datasets/benjaminpo/finance-dataset-intraday) | `1m` … `1h` (dated snapshots) |

It shows how to:
1. Locate both attached dataset roots on Kaggle (or a local `data/` checkout)
2. Load cumulative daily bars, plot **normalized prices**, and run a **hybrid ARIMA–LSTM** price forecast
3. Load dated intradaily snapshots and plot an intraday session

Pipeline / listings: [benjaminpo/finance-dataset](https://github.com/benjaminpo/finance-dataset)
3. Load dated intradaily snapshots and plot an intraday session

Pipeline / listings: [benjaminpo/finance-dataset](https://github.com/benjaminpo/finance-dataset)

## 1. Locate daily + intraday roots

On Kaggle the mounts are often nested as `/kaggle/input/datasets/<user>/<slug>/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.options.display.max_rows = 8
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

ASSET_MARKERS = (
    "stocks_us",
    "stocks_kr",
    "stocks_jp",
    "stocks_eu",
    "stocks_hk",
    "indices",
    "rates",
    "futures",
    "crypto",
    "currencies",
)
DAILY_INTERVALS = {"1d", "5d", "1wk", "1mo", "3mo"}
INTRADAY_INTERVALS = {"1m", "2m", "5m", "15m", "30m", "60m", "90m", "1h"}


def _looks_like_data_root(path: Path) -> bool:
    return path.is_dir() and any((path / asset).is_dir() for asset in ASSET_MARKERS)


def _iter_kaggle_candidates() -> list[Path]:
    kaggle_input = Path("/kaggle/input")
    if not kaggle_input.is_dir():
        print("/kaggle/input not present (local run)")
        return []

    mounted = sorted(p for p in kaggle_input.iterdir() if p.is_dir())
    print("Mounted under /kaggle/input:", [p.name for p in mounted] or "(none)")

    found: list[Path] = []
    frontier = list(mounted)
    for _ in range(4):
        nxt: list[Path] = []
        for root in frontier:
            if _looks_like_data_root(root):
                found.append(root)
                continue
            try:
                nxt.extend(sorted(p for p in root.iterdir() if p.is_dir()))
            except OSError:
                pass
        frontier = nxt
    return found


def _interval_dirs(root: Path) -> set[str]:
    names: set[str] = set()
    for asset in ASSET_MARKERS:
        asset_dir = root / asset
        if not asset_dir.is_dir():
            continue
        for child in asset_dir.iterdir():
            if child.is_dir():
                names.add(child.name)
    return names


def _score_daily(root: Path) -> int:
    intervals = _interval_dirs(root)
    score = len(intervals & DAILY_INTERVALS) * 10
    # Prefer the dedicated daily slug when both trees are present.
    name = root.name.lower()
    if "intraday" in name:
        score -= 50
    elif name.endswith("finance-dataset") or name == "finance-dataset":
        score += 20
    return score


def _score_intraday(root: Path) -> int:
    intervals = _interval_dirs(root)
    score = len(intervals & INTRADAY_INTERVALS) * 10
    name = root.name.lower()
    if "intraday" in name:
        score += 50
    return score


def find_data_roots() -> tuple[Path, Path | None]:
    """Return (daily_root, intraday_root). Intraday may be None locally."""
    candidates = _iter_kaggle_candidates()

    here = Path.cwd()
    local = [
        here / "data",
        here.parent / "data",
        here.parent.parent / "data",
    ]
    candidates.extend(p for p in local if _looks_like_data_root(p))

    # Deduplicate while preserving order.
    uniq: list[Path] = []
    seen: set[Path] = set()
    for path in candidates:
        resolved = path.resolve() if path.exists() else path
        if resolved in seen or not _looks_like_data_root(path):
            continue
        seen.add(resolved)
        uniq.append(path)

    if not uniq:
        raise FileNotFoundError(
            "Could not find dataset root(s). Attach benjaminpo/finance-dataset and "
            "benjaminpo/finance-dataset-intraday on Kaggle, or use a local data/ dir."
        )

    daily = max(uniq, key=_score_daily)
    intraday_ranked = sorted(uniq, key=_score_intraday, reverse=True)
    intraday = intraday_ranked[0] if _score_intraday(intraday_ranked[0]) > 0 else None
    if intraday is not None and intraday.resolve() == daily.resolve():
        # Same tree (combined local checkout / legacy upload): reuse it.
        pass
    elif intraday is not None and "intraday" not in intraday.name.lower():
        # Only keep a non-named tree if it actually has intradaily intervals.
        if not (_interval_dirs(intraday) & INTRADAY_INTERVALS):
            intraday = None

    return daily, intraday


DAILY_DIR, INTRADAY_DIR = find_data_roots()
print("DAILY_DIR   =", DAILY_DIR)
print("INTRADAY_DIR=", INTRADAY_DIR)

## 2. Daily layout + load helpers

- **Cumulative** (`1d`, `1wk`, …): `{asset_class}/{interval}/{TICKER}.csv`
- Yahoo tickers like `^GSPC` / `EURUSD=X` become `GSPC.csv` / `EURUSD_X.csv` on disk.

In [ ]:
PREFERRED_DAILY = ("1d", "1wk", "1mo", "3mo", "5d")

asset_classes = sorted(
    p.name for p in DAILY_DIR.iterdir() if p.is_dir() and not p.name.startswith(".")
)
print("Daily asset classes:", ", ".join(asset_classes))

rows = []
for asset_class in asset_classes:
    asset_dir = DAILY_DIR / asset_class
    for name in PREFERRED_DAILY:
        interval_dir = asset_dir / name
        if not interval_dir.is_dir():
            continue
        sample = []
        for path in interval_dir.iterdir():
            if path.suffix == ".csv":
                sample.append(path.name)
                if len(sample) >= 3:
                    break
        if sample:
            rows.append(
                {
                    "asset_class": asset_class,
                    "interval": name,
                    "sample_files": ", ".join(sample),
                }
            )

pd.DataFrame(rows)

In [ ]:
def safe_filename(ticker: str) -> str:
    """Match the pipeline's on-disk naming (^GSPC → GSPC.csv, EURUSD=X → EURUSD_X.csv)."""
    return ticker.replace("^", "").replace("=", "_").replace("/", "_")


def load_ohlcv(
    asset_class: str,
    ticker: str,
    interval: str = "1d",
    *,
    data_dir: Path = None,
) -> pd.DataFrame:
    root = DAILY_DIR if data_dir is None else data_dir
    path = root / asset_class / interval / f"{safe_filename(ticker)}.csv"
    return pd.read_csv(path, parse_dates=["Datetime"], index_col="Datetime").sort_index()


def load_close(
    asset_class: str,
    ticker: str,
    interval: str = "1d",
    *,
    data_dir: Path = None,
) -> pd.Series:
    df = load_ohlcv(asset_class, ticker, interval=interval, data_dir=data_dir)
    col = "Adj Close" if "Adj Close" in df.columns else "Close"
    s = df[col].astype(float)
    s.name = ticker
    return s


aapl = load_ohlcv("stocks_us", "AAPL")
print(f"AAPL daily: {aapl.index.min().date()} → {aapl.index.max().date()} ({len(aapl)} bars)")
aapl.tail()

## 3. Daily multi-asset prices & returns

## 3. Daily multi-asset prices & returns

Normalized price (`P_t / P_0`) is the same curve as compounding daily simple returns from $1 — so we plot it once, then forecast with a hybrid ARIMA–LSTM in the next section.

In [ ]:
SERIES = [
    ("stocks_us", "AAPL"),
    ("stocks_us", "MSFT"),
    ("indices", "^GSPC"),
    ("crypto", "BTC-USD"),
    ("currencies", "EURUSD=X"),
]

closes = pd.concat(
    [load_close(asset_class, ticker) for asset_class, ticker in SERIES],
    axis=1,
).dropna(how="any")

LOOKBACK_DAYS = 365 * 3
window = closes.iloc[-LOOKBACK_DAYS:]
returns = window.pct_change().dropna(how="any")
# Same shape as window / window.iloc[0] (growth of $1 from the first bar).
normalized = window.div(window.iloc[0])

print(
    f"Window: {window.index.min().date()} → {window.index.max().date()} "
    f"({len(window)} bars)"
)
returns.describe().T[["mean", "std", "min", "max"]]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
normalized.plot(ax=ax, lw=1.4)
ax.set_title("Normalized price (start = 1.0) — same as growth of $1")
ax.set_ylabel("Index")
ax.set_xlabel("Datetime (UTC)")
ax.legend(loc="upper left", fontsize=9)
plt.show()

In [ ]:
corr = returns.corr()

fig, ax = plt.subplots(figsize=(5.5, 4.5), constrained_layout=True)
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.index)
ax.set_title("Daily return correlation")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=8)

plt.show()
corr

## 4. Hybrid ARIMA–LSTM price forecast

Demo only — not investment advice.

**Idea:** ARIMA captures the **linear** autocorrelation in log-price; an LSTM learns the leftover **non-linear** residual pattern. Final forecast:

$$\hat{y}_t = \underbrace{\hat{y}^{\mathrm{ARIMA}}_t}_{\text{linear}} + \underbrace{\hat{e}^{\mathrm{LSTM}}_t}_{\text{residual}}$$

We pick a small ARIMA order by AIC on the train window, train the LSTM only on train residuals (no test leakage), then roll one-step ahead through the holdout.

In [ ]:
import os
import warnings
from itertools import product

# Force CPU before importing TF (Kaggle CPU sessions still probe CUDA otherwise).
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

import tensorflow as tf
from statsmodels.tsa.arima.model import ARIMA
from tensorflow import keras
from tensorflow.keras import layers

try:
    tf.config.set_visible_devices([], "GPU")
except Exception:
    pass

FORECAST_TICKER = "AAPL"
TEST_DAYS = 90
LOOKBACK = 20
EPOCHS = 30
BATCH_SIZE = 32
LSTM_UNITS = 32
SEED = 42

tf.keras.utils.set_random_seed(SEED)

price = window[FORECAST_TICKER].astype(float)
log_price = np.log(price)
split = len(log_price) - TEST_DAYS
log_train = log_price.iloc[:split]
log_test = log_price.iloc[split:]
# Integer index avoids statsmodels append failures on gappy trading-day calendars.
log_train_ri = pd.Series(log_train.to_numpy(dtype=float))
log_test_vals = log_test.to_numpy(dtype=float)


def _arima_fit(series: pd.Series, order: tuple[int, int, int]):
    """Fit ARIMA; return None if MLE did not converge."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fit = ARIMA(series, order=order).fit(method_kwargs={"maxiter": 200})
    mle = getattr(fit, "mle_retvals", None) or {}
    if mle and not mle.get("converged", True):
        return None
    if not np.isfinite(getattr(fit, "aic", np.nan)):
        return None
    return fit


def select_arima_order(series: pd.Series, max_p: int = 2, max_q: int = 2) -> tuple[int, int, int]:
    """AIC search over a small (p,1,q) grid — keeps the demo fast."""
    best_order, best_aic = (1, 1, 0), np.inf
    for p, q in product(range(max_p + 1), range(max_q + 1)):
        if p == 0 and q == 0:
            continue
        try:
            fit = _arima_fit(series, (p, 1, q))
            if fit is not None and fit.aic < best_aic:
                best_aic, best_order = fit.aic, (p, 1, q)
        except Exception:
            continue
    return best_order


def make_residual_windows(resid: np.ndarray, lookback: int) -> tuple[np.ndarray, np.ndarray]:
    x, y = [], []
    for i in range(lookback, len(resid)):
        x.append(resid[i - lookback : i])
        y.append(resid[i])
    x = np.asarray(x, dtype=np.float32)[..., np.newaxis]
    y = np.asarray(y, dtype=np.float32)
    return x, y


def build_lstm(lookback: int, units: int = LSTM_UNITS) -> keras.Model:
    model = keras.Sequential(
        [
            layers.Input(shape=(lookback, 1)),
            layers.LSTM(units, return_sequences=False),
            layers.Dense(units // 2, activation="relu"),
            layers.Dense(1),
        ]
    )
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    return model


order = select_arima_order(log_train_ri)
print(
    f"{FORECAST_TICKER}: ARIMA{order} selected by AIC on train "
    f"({log_train.index.min().date()} → {log_train.index.max().date()})"
)

arima_res = _arima_fit(log_train_ri, order)
if arima_res is None:
    # Fall back to a simple, usually well-behaved order.
    order = (1, 1, 0)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        arima_res = ARIMA(log_train_ri, order=order).fit(method_kwargs={"maxiter": 500})
    print(f"Falling back to ARIMA{order} after non-convergence")

train_resid = arima_res.resid.dropna().to_numpy(dtype=np.float64)

x_tr, y_tr = make_residual_windows(train_resid, LOOKBACK)
lstm = build_lstm(LOOKBACK)
lstm.fit(
    x_tr,
    y_tr,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    verbose=0,
    shuffle=False,
)

# Roll one-step ARIMA (append, no refit) + LSTM residual correction through the holdout.
arima_state = arima_res
resid_hist = list(train_resid)
arima_log_preds: list[float] = []
hybrid_log_preds: list[float] = []
next_ix = len(log_train_ri)

for t, actual_log in enumerate(log_test_vals):
    arima_hat = float(arima_state.forecast(steps=1).iloc[0])
    window_resid = np.asarray(resid_hist[-LOOKBACK:], dtype=np.float32).reshape(1, LOOKBACK, 1)
    lstm_hat = float(lstm.predict(window_resid, verbose=0).ravel()[0])
    hybrid_hat = arima_hat + lstm_hat

    arima_log_preds.append(arima_hat)
    hybrid_log_preds.append(hybrid_hat)

    # Observed ARIMA residual for the next LSTM window (available after the close).
    resid_hist.append(float(actual_log) - arima_hat)
    arima_state = arima_state.append(
        pd.Series([float(actual_log)], index=[next_ix + t]),
        refit=False,
    )

# Back to price space.
actual = price.iloc[split:].rename("actual")
arima_pred = pd.Series(np.exp(arima_log_preds), index=actual.index, name="arima")
predicted = pd.Series(np.exp(hybrid_log_preds), index=actual.index, name="hybrid")
err = predicted - actual
arima_err = arima_pred - actual


def _metrics(pred: pd.Series) -> dict[str, float]:
    e = pred - actual
    return {
        "MAE": float(e.abs().mean()),
        "RMSE": float(np.sqrt((e**2).mean())),
        "MAPE%": float((e.abs() / actual).mean() * 100),
    }


naive = price.shift(1).loc[actual.index]
metrics = pd.DataFrame(
    {
        "naive": _metrics(naive.rename("naive")),
        "ARIMA": _metrics(arima_pred),
        "ARIMA+LSTM": _metrics(predicted),
    }
).T

print(
    f"Holdout: {actual.index.min().date()} → {actual.index.max().date()} "
    f"({len(actual)} days) | LSTM lookback={LOOKBACK}, epochs={EPOCHS}"
)
metrics


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9), constrained_layout=True)

# Draw ARIMA on top with a dashed stroke — one-step ARIMA≈hybrid on price scale,
# so a solid overlap makes ARIMA look "missing".
axes[0].plot(actual.index, actual.values, lw=2.0, label="Actual", color="0.25", zorder=1)
axes[0].plot(
    predicted.index,
    predicted.values,
    lw=1.6,
    label="ARIMA+LSTM",
    color="C2",
    alpha=0.95,
    zorder=2,
)
axes[0].plot(
    arima_pred.index,
    arima_pred.values,
    lw=1.8,
    ls="--",
    label="ARIMA",
    color="C1",
    zorder=3,
)
axes[0].set_title(
    f"{FORECAST_TICKER} — hybrid ARIMA{order}+LSTM vs actual ({TEST_DAYS}-day holdout)"
)
axes[0].set_ylabel("Price")
axes[0].legend(loc="upper left", fontsize=9)

axes[1].plot(
    arima_err.index, arima_err.values, lw=1.4, ls="--", color="C1", label="ARIMA error", zorder=3
)
axes[1].plot(err.index, err.values, lw=1.2, color="C2", label="Hybrid error", zorder=2)
axes[1].axhline(0.0, color="k", lw=0.8, alpha=0.5)
axes[1].set_title("Prediction error (predicted − actual)")
axes[1].set_ylabel("Error")
axes[1].legend(loc="upper left", fontsize=9)

# Price-space gap between the two models (what the LSTM actually changes).
hybrid_minus_arima = predicted - arima_pred
axes[2].plot(hybrid_minus_arima.index, hybrid_minus_arima.values, lw=1.2, color="C4")
axes[2].axhline(0.0, color="k", lw=0.8, alpha=0.5)
axes[2].set_title("LSTM contribution in price space (hybrid − ARIMA)")
axes[2].set_ylabel("Δ price")
axes[2].set_xlabel("Datetime (UTC)")

print(
    f"Mean |hybrid − ARIMA| = {hybrid_minus_arima.abs().mean():.4f} "
    f"(max {hybrid_minus_arima.abs().max():.4f}) — small values mean the lines nearly overlap"
)
plt.show()
metrics

## 5. Intraday snapshots

Yahoo only keeps a short rolling window, so intradaily bars are stored as **dated files**:

`{asset_class}/{interval}/{TICKER}_{YYYY-MM-DD}.csv`

Example: `stocks_us/5m/AAPL_2026-07-21.csv`. Concatenate several days to rebuild a longer series.

In [ ]:
if INTRADAY_DIR is None:
    raise FileNotFoundError(
        "Intraday dataset not found. Attach benjaminpo/finance-dataset-intraday "
        "(or use a local data/ tree that still contains 1m…1h folders)."
    )


def snapshot_path(
    asset_class: str,
    ticker: str,
    day: str,
    interval: str = "5m",
    *,
    data_dir: Path | None = None,
) -> Path:
    root = INTRADAY_DIR if data_dir is None else data_dir
    return root / asset_class / interval / f"{safe_filename(ticker)}_{day}.csv"


def recent_snapshot_days(
    asset_class: str,
    ticker: str,
    interval: str = "5m",
    *,
    lookback_calendar_days: int = 21,
    data_dir: Path | None = None,
) -> list[str]:
    """Probe recent calendar dates instead of listing huge snapshot folders."""
    today = pd.Timestamp.utcnow().normalize()
    days: list[str] = []
    for offset in range(lookback_calendar_days):
        day = (today - pd.Timedelta(days=offset)).strftime("%Y-%m-%d")
        if snapshot_path(asset_class, ticker, day, interval, data_dir=data_dir).is_file():
            days.append(day)
    return sorted(days)


def load_intraday(
    asset_class: str,
    ticker: str,
    interval: str = "5m",
    *,
    days: list[str] | None = None,
    last_n_days: int | None = 5,
    lookback_calendar_days: int = 21,
    data_dir: Path | None = None,
) -> pd.DataFrame:
    """Load and concatenate dated intradaily snapshot CSVs."""
    root = INTRADAY_DIR if data_dir is None else data_dir
    available = days or recent_snapshot_days(
        asset_class,
        ticker,
        interval,
        lookback_calendar_days=lookback_calendar_days,
        data_dir=root,
    )
    if not available:
        raise FileNotFoundError(
            f"No {interval} snapshots for {ticker} under {root / asset_class / interval} "
            f"in the last {lookback_calendar_days} calendar days"
        )

    if days is None and last_n_days is not None:
        available = available[-last_n_days:]

    frames = [
        pd.read_csv(
            snapshot_path(asset_class, ticker, day, interval, data_dir=root),
            parse_dates=["Datetime"],
            index_col="Datetime",
        )
        for day in available
    ]
    df = pd.concat(frames).sort_index()
    return df[~df.index.duplicated(keep="last")]


# Prefer US equity 5m; fall back to BTC if needed.
INTRADAY_CANDIDATES = [
    ("stocks_us", "AAPL", "5m"),
    ("stocks_us", "MSFT", "5m"),
    ("crypto", "BTC-USD", "5m"),
    ("crypto", "BTC-USD", "15m"),
]

intraday_meta = None
for asset_class, ticker, interval in INTRADAY_CANDIDATES:
    days = recent_snapshot_days(asset_class, ticker, interval)
    if days:
        intraday_meta = (asset_class, ticker, interval, days)
        break

if intraday_meta is None:
    raise FileNotFoundError(
        f"No recent intradaily snapshots under {INTRADAY_DIR} for {INTRADAY_CANDIDATES}"
    )

asset_class, ticker, interval, found_days = intraday_meta
print(
    f"Using {asset_class}/{interval}/{safe_filename(ticker)}_*.csv — "
    f"found {len(found_days)} day(s) in lookback, latest={found_days[-1]}"
)
print("Days:", ", ".join(found_days))

intraday = load_intraday(asset_class, ticker, interval, days=found_days[-5:])
print(
    f"Loaded {len(intraday)} bars: "
    f"{intraday.index.min()} → {intraday.index.max()}"
)
intraday.tail()


In [ ]:
price_col = "Adj Close" if "Adj Close" in intraday.columns else "Close"
latest_day = str(intraday.index.max().date())
session = intraday.loc[latest_day]

fig, axes = plt.subplots(2, 1, figsize=(10, 7), constrained_layout=True)

intraday[price_col].plot(ax=axes[0], lw=1.2, color="C0")
axes[0].set_title(f"{ticker} {interval} — last {intraday.index.normalize().nunique()} session(s)")
axes[0].set_ylabel(price_col)
axes[0].set_xlabel("")

session[price_col].plot(ax=axes[1], lw=1.4, color="C1")
axes[1].set_title(f"{ticker} {interval} — {latest_day} only")
axes[1].set_ylabel(price_col)
axes[1].set_xlabel("Datetime (UTC)")

plt.show()

# Simple intradaily return stats for the loaded window.
intra_ret = intraday[price_col].pct_change().dropna()
pd.Series(
    {
        "bars": len(intraday),
        "sessions": int(intraday.index.normalize().nunique()),
        "mean_ret": float(intra_ret.mean()),
        "std_ret": float(intra_ret.std()),
        "min_ret": float(intra_ret.min()),
        "max_ret": float(intra_ret.max()),
    },
    name=f"{ticker} {interval}",
).to_frame("value")

## 6. Path cheat sheet

| Example | Dataset | Path |
|---|---|---|
| Apple daily | daily | `stocks_us/1d/AAPL.csv` |
| Apple weekly | daily | `stocks_us/1wk/AAPL.csv` |
| Bitcoin daily | daily | `crypto/1d/BTC-USD.csv` |
| S&P 500 | daily | `indices/1d/GSPC.csv` (`^` stripped) |
| EUR/USD | daily | `currencies/1d/EURUSD_X.csv` (`=` → `_`) |
| AAPL 5-minute day | intraday | `stocks_us/5m/AAPL_YYYY-MM-DD.csv` |
| BTC 1-minute day | intraday | `crypto/1m/BTC-USD_YYYY-MM-DD.csv` |

Asset classes: `stocks_us`, `stocks_kr`, `stocks_jp`, `stocks_eu`, `stocks_hk`, `indices`, `rates`, `futures`, `crypto`, `currencies`.